In [8]:
def extract_averaged_seizure_epochs_by_band(nwb_file: Path, 
                                            seizure_df: pd.DataFrame,
                                            valid_channel_indices: set,
                                            window_minutes: int = 7) -> Dict:
    """
    Extract bandpower time series averaged across valid channels, SEPARATELY for each band.
    
    Returns:
        dict: {seizure_idx: {band_name: (time_array, avg_bandpower_array)}}
    """
    seizure_data = {}
    
    try:
        with NWBHDF5IO(str(nwb_file), 'r') as io:
            nwb = io.read()
            
            # Get session start time
            session_start = nwb.session_start_time
            
            # Load band power data
            band_power_series = nwb.processing['ecephys']["band_power"]
            band_power_data = band_power_series.data[:]  # Shape: (timepoints, channels, bands)
            
            # Get timestamps
            if band_power_series.timestamps is not None:
                band_power_timestamps = band_power_series.timestamps[:]
            else:
                starting_time = band_power_series.starting_time
                rate = band_power_series.rate
                n_timepoints = band_power_data.shape[0]
                band_power_timestamps = starting_time + np.arange(n_timepoints) / rate
            
            # Convert timestamps to datetime
            band_power_datetimes = pd.to_datetime(session_start) + pd.to_timedelta(band_power_timestamps, unit='s')
            band_power_datetimes = band_power_datetimes.tz_localize(None)
            
            # Iterate through each seizure event
            for seizure_idx, row in seizure_df.iterrows():
                seizure_time = row['date']
                
                # Calculate window
                window_start = seizure_time - pd.Timedelta(minutes=window_minutes)
                window_end = seizure_time + pd.Timedelta(minutes=window_minutes)
                
                # Find indices within the window
                mask = (band_power_datetimes >= window_start) & (band_power_datetimes <= window_end)
                
                if not np.any(mask):
                    continue
                
                # Get data for this window
                window_data = band_power_data[mask, :, :]  # Shape: (time_in_window, channels, bands)
                window_times = band_power_datetimes[mask]
                
                # Filter to valid channels only
                valid_channel_list = sorted(list(valid_channel_indices))
                window_data_filtered = window_data[:, valid_channel_list, :]  # Shape: (time, valid_channels, bands)
                
                # Convert times to minutes relative to seizure
                time_relative = (window_times - seizure_time).total_seconds() / 60.0
                time_relative = time_relative.values
                
                # Store data separately for each band
                if seizure_idx not in seizure_data:
                    seizure_data[seizure_idx] = {band: {'times': [], 'powers': []} for band in BAND_NAMES}
                
                # Average across channels (but NOT across bands)
                for band_idx, band_name in enumerate(BAND_NAMES):
                    avg_bandpower = np.mean(window_data_filtered[:, :, band_idx], axis=1)  # Shape: (time,)
                    
                    seizure_data[seizure_idx][band_name]['times'].append(time_relative)
                    seizure_data[seizure_idx][band_name]['powers'].append(avg_bandpower)
            
    except Exception as e:
        print(f"  Error processing {nwb_file.name}: {e}")
        return {}
    
    return seizure_data


def load_all_participants_data(file_metadata: pd.DataFrame,
                               window_minutes: int = 7) -> List[Dict]:
    """
    Load averaged seizure epoch data for all participants.
    Data is averaged across channels but kept separate for each frequency band.
    
    Returns:
        List of dicts with participant data
    """
    
    print(f"\n{'='*60}")
    print(f"Finding all subjects with seizure data...")
    print(f"{'='*60}\n")
    
    # Find all subjects
    subjects_sessions = get_all_subjects_with_seizures(file_metadata)
    
    if len(subjects_sessions) == 0:
        print("No subjects found with preprocessed data and seizure files!")
        return []
    
    print(f"\nFound {len(subjects_sessions)} subject-session pairs")
    print(f"{'='*60}\n")
    
    # Load data for all subjects
    all_participant_data = []
    
    for subject_id, session_id in subjects_sessions:
        print(f"Loading sub-{subject_id}, ses-{session_id}...")
        
        # Load seizure times
        seizure_df = load_seizure_times(subject_id, session_id)
        
        if len(seizure_df) == 0:
            print(f"  No seizure events found")
            continue
        
        # Get only relevant files that overlap with seizure windows
        relevant_files = get_relevant_files(
            file_metadata, subject_id, session_id, seizure_df, window_minutes
        )
        
        if len(relevant_files) == 0:
            print(f"  No relevant files found")
            continue
        
        print(f"  Found {len(relevant_files)} relevant files (out of all session files)")
        
        # Load and filter electrode labels from first file
        electrode_labels = load_electrode_labels(relevant_files[0])
        valid_electrode_labels = filter_valid_channels(electrode_labels)
        valid_channel_indices = set(valid_electrode_labels.keys())
        
        n_valid_channels = len(valid_channel_indices)
        
        if n_valid_channels == 0:
            print(f"  No valid channels")
            continue
        
        # Collect averaged data across relevant files only
        all_seizure_data = {}
        
        for nwb_file in relevant_files:
            file_data = extract_averaged_seizure_epochs_by_band(
                nwb_file, seizure_df, valid_channel_indices, window_minutes
            )
            
            # Merge data
            for seizure_idx, band_dict in file_data.items():
                if seizure_idx not in all_seizure_data:
                    all_seizure_data[seizure_idx] = {band: {'times': [], 'powers': []} for band in BAND_NAMES}
                
                for band_name in BAND_NAMES:
                    all_seizure_data[seizure_idx][band_name]['times'].extend(band_dict[band_name]['times'])
                    all_seizure_data[seizure_idx][band_name]['powers'].extend(band_dict[band_name]['powers'])
        
        # Concatenate data for each seizure and band
        for seizure_idx in all_seizure_data.keys():
            for band_name in BAND_NAMES:
                times_list = all_seizure_data[seizure_idx][band_name]['times']
                powers_list = all_seizure_data[seizure_idx][band_name]['powers']
                
                if len(times_list) > 0:
                    all_times = np.concatenate(times_list)
                    all_powers = np.concatenate(powers_list)
                    
                    # Sort by time
                    sort_idx = np.argsort(all_times)
                    all_seizure_data[seizure_idx][band_name] = (all_times[sort_idx], all_powers[sort_idx])
                else:
                    all_seizure_data[seizure_idx][band_name] = (np.array([]), np.array([]))
        
        if len(all_seizure_data) > 0:
            all_participant_data.append({
                'subject_id': subject_id,
                'session_id': session_id,
                'seizure_data': all_seizure_data,
                'seizure_df': seizure_df,
                'n_valid_channels': n_valid_channels,
                'n_files_used': len(relevant_files)
            })
            print(f"  Loaded {len(all_seizure_data)} seizure(s), {n_valid_channels} valid channels")
        else:
            print(f"  No valid seizure data")
    
    print(f"\n{'='*60}")
    print(f"Successfully loaded data for {len(all_participant_data)} participants")
    print(f"{'='*60}\n")
    
    return all_participant_data




In [9]:
#!/usr/bin/env python3
"""
Updated seizure epoch plotting with broadband power option
"""

# Add this constant near the top with your other constants
BAND_NAMES = ['delta', 'theta', 'alpha', 'beta', 'gamma', 'high_gamma']

# Define bandwidths for each frequency band
BAND_WIDTHS = {
    'delta': 3,        # 1-4 Hz
    'theta': 4,        # 4-8 Hz
    'alpha': 5,        # 8-13 Hz
    'beta': 17,        # 13-30 Hz
    'gamma': 50,       # 30-80 Hz
    'high_gamma': 120  # 80-200 Hz
}


def compute_broadband_power(band_dict: Dict, method: str = 'weighted') -> tuple:
    """
    Compute broadband power from individual frequency bands.
    
    Args:
        band_dict: Dictionary with band_name -> (time_array, power_array)
        method: 'weighted' (bandwidth-weighted), 'mean' (simple average), or 'geomean'
    
    Returns:
        (time_array, broadband_power_array)
    """
    # Collect all band powers and align times
    band_powers = {}
    time_arrays = {}
    
    for band_name in BAND_NAMES:
        if band_name in band_dict:
            time_array, power_array = band_dict[band_name]
            if len(time_array) > 0:
                band_powers[band_name] = power_array
                time_arrays[band_name] = time_array
    
    if len(band_powers) == 0:
        return np.array([]), np.array([])
    
    # Use the first band's time array (should all be the same)
    time_ref = time_arrays[list(time_arrays.keys())[0]]
    
    # Verify all time arrays are the same
    for band_name, time_array in time_arrays.items():
        if not np.array_equal(time_array, time_ref):
            print(f"Warning: Time arrays differ for {band_name}. Using first band's times.")
            break
    
    # Compute broadband power
    if method == 'weighted':
        # Bandwidth-weighted average
        total_bandwidth = sum(BAND_WIDTHS[b] for b in band_powers.keys())
        broadband = np.zeros_like(list(band_powers.values())[0])
        
        for band_name, power_array in band_powers.items():
            weight = BAND_WIDTHS[band_name] / total_bandwidth
            broadband += power_array * weight
    
    elif method == 'mean':
        # Simple arithmetic mean
        broadband = np.mean(np.stack(list(band_powers.values())), axis=0)
    
    elif method == 'geomean':
        # Geometric mean (for multiplicative data)
        from scipy.stats import gmean
        broadband = gmean(np.stack(list(band_powers.values())), axis=0)
    
    else:
        raise ValueError(f"Unknown method: {method}")
    
    return time_ref, broadband


def plot_all_participants(all_participant_data: List[Dict],
                         output_dir: str = './plots',
                         window_minutes: int = 7,
                         n_cols: int = 3,
                         bands_to_plot: List[str] = None,
                         log_scale: bool = True,
                         include_broadband: bool = False,
                         broadband_method: str = 'weighted'):
    """
    Plot all participants' bandpower around seizures.
    
    - If plotting ONE band: different seizures get different colors
    - If plotting MULTIPLE bands: different bands get different colors, different seizures get different line styles
    
    Args:
        all_participant_data: List of dicts with participant data (from load_all_participants_data)
        output_dir: Directory to save plots
        window_minutes: Window size (for title)
        n_cols: Number of columns in subplot grid
        bands_to_plot: List of bands to plot (default: all bands)
        log_scale: If True, use log scale for y-axis (default: True)
        include_broadband: If True, add broadband power trace (default: False)
        broadband_method: Method for computing broadband ('weighted', 'mean', or 'geomean')
    """
    
    if len(all_participant_data) == 0:
        print("No data to plot!")
        return
    
    if bands_to_plot is None:
        bands_to_plot = BAND_NAMES
    
    single_band_mode = len(bands_to_plot) == 1 and not include_broadband
    
    print(f"\n{'='*60}")
    print(f"Creating plot for {len(all_participant_data)} participants...")
    if single_band_mode:
        print(f"Single band mode: {bands_to_plot[0]}")
    elif include_broadband and len(bands_to_plot) == 0:
        print(f"Broadband only mode (method: {broadband_method})")
    else:
        print(f"Multi-band mode: {len(bands_to_plot)} bands")
        if include_broadband:
            print(f"  + Broadband (method: {broadband_method})")
    print(f"{'='*60}\n")
    
    # Calculate grid dimensions
    n_participants = len(all_participant_data)
    n_rows = int(np.ceil(n_participants / n_cols))
    
    # Create figure
    fig_width = n_cols * 6
    fig_height = n_rows * 4
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height))
    axes = np.atleast_2d(axes).flatten()
    
    # Color map for bands (used in multi-band mode)
    band_colors = {
        'delta': '#1f77b4',      # blue
        'theta': '#ff7f0e',      # orange
        'alpha': '#2ca02c',      # green
        'beta': '#d62728',       # red
        'gamma': '#9467bd',      # purple
        'high_gamma': '#8c564b', # brown
        'broadband': '#000000'   # black for broadband
    }
    
    # Plot each participant
    for plot_idx, participant_data in enumerate(all_participant_data):
        ax = axes[plot_idx]
        
        subject_id = participant_data['subject_id']
        session_id = participant_data['session_id']
        seizure_data = participant_data['seizure_data']
        n_valid_channels = participant_data['n_valid_channels']
        n_files_used = participant_data.get('n_files_used', '?')
        
        n_seizures = len(seizure_data)
        
        if single_band_mode:
            # SINGLE BAND MODE: Different colors for different seizures
            seizure_colors = plt.cm.Set1(np.linspace(0, 1, min(n_seizures, 9)))
            if n_seizures > 9:
                seizure_colors = plt.cm.tab20(np.linspace(0, 1, n_seizures))
            
            band_name = bands_to_plot[0]
            
            for seizure_idx_pos, (seizure_idx, band_dict) in enumerate(seizure_data.items()):
                if band_name not in band_dict.keys():
                    continue
                
                time_array, power_array = band_dict[band_name]
                
                if len(time_array) == 0:
                    continue
                
                label = f'Seizure {seizure_idx_pos + 1}'
                
                ax.plot(time_array, power_array,
                       color=seizure_colors[seizure_idx_pos],
                       linestyle='-',
                       alpha=0.7,
                       linewidth=2,
                       label=label)
        
        else:
            # MULTI-BAND MODE: Different colors for bands, different line styles for seizures
            linestyles = ['-', '--', '-.', ':']
            
            for seizure_idx_pos, (seizure_idx, band_dict) in enumerate(seizure_data.items()):
                linestyle = linestyles[seizure_idx_pos % len(linestyles)]
                
                # Plot each band
                for band_name in bands_to_plot:
                    if band_name not in band_dict.keys():
                        continue
                    
                    time_array, power_array = band_dict[band_name]
                    
                    if len(time_array) == 0:
                        continue
                    
                    label = f'{band_name.capitalize()}'
                    if n_seizures > 1:
                        label += f' (S{seizure_idx_pos + 1})'
                    
                    ax.plot(time_array, power_array,
                           color=band_colors.get(band_name, 'gray'),
                           linestyle=linestyle,
                           alpha=0.7,
                           linewidth=2,
                           label=label if seizure_idx_pos == 0 else '')  # Only label first seizure
                
                # Add broadband if requested
                if include_broadband:
                    time_array, broadband_power = compute_broadband_power(band_dict, method=broadband_method)
                    
                    if len(time_array) > 0:
                        label = 'Broadband'
                        if n_seizures > 1:
                            label += f' (S{seizure_idx_pos + 1})'
                        
                        ax.plot(time_array, broadband_power,
                               color=band_colors['broadband'],
                               linestyle=linestyle,
                               alpha=0.9,
                               linewidth=3,
                               label=label if seizure_idx_pos == 0 else '',
                               zorder=10)  # Plot on top
        
        # Mark seizure time
        ax.axvline(0, color='red', linestyle='--', linewidth=2, alpha=0.5, zorder=0)
        
        # Formatting
        ax.set_xlabel('Time relative to seizure (min)', fontsize=9)
        ylabel = 'Avg Bandpower (μV²/Hz)'
        if log_scale:
            ylabel += ' [log scale]'
        ax.set_ylabel(ylabel, fontsize=9)
        
        title = f'Sub-{subject_id}, Ses-{session_id}\n{n_seizures} seizure(s), {n_valid_channels} channels, {n_files_used} files'
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        # Set scale
        if log_scale:
            ax.set_yscale('log')
        
        # Add legend (only if there's something to show)
        if n_seizures > 1 or not single_band_mode or include_broadband:
            ax.legend(loc='upper right', fontsize=7, framealpha=0.9)
    
    # Remove empty subplots
    for ax in axes[len(all_participant_data):]:
        ax.remove()
    
    # Overall title
    scale_text = ' [Log Scale]' if log_scale else ''
    if single_band_mode:
        band_text = f' - {bands_to_plot[0].capitalize()} Band'
    elif include_broadband and len(bands_to_plot) == 0:
        band_text = f' - Broadband ({broadband_method})'
    else:
        band_text = f' by Frequency Band'
        if include_broadband:
            band_text += f' + Broadband ({broadband_method})'
    
    fig.suptitle(f'Bandpower Around Seizure Events{band_text}{scale_text} (All Participants)\n'
                 f'Averaged across valid channels | ±{window_minutes} min windows',
                 fontsize=14, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    
    # Save plot
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    suffix = '_log' if log_scale else '_linear'
    if single_band_mode:
        suffix += f'_{bands_to_plot[0]}'
    elif include_broadband and len(bands_to_plot) == 0:
        suffix += f'_broadband_{broadband_method}'
    elif include_broadband:
        suffix += f'_with_broadband_{broadband_method}'
    
    output_file = Path(output_dir) / f'all_participants_seizure_epochs{suffix}.png'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"\nSaved: {output_file}")
    plt.show()
    plt.close()


def plot_broadband_only(all_participant_data: List[Dict],
                       output_dir: str = './plots',
                       window_minutes: int = 7,
                       n_cols: int = 3,
                       log_scale: bool = True,
                       broadband_method: str = 'weighted'):
    """
    Convenience function to plot ONLY broadband power (no individual bands).
    Different seizures get different colors.
    
    Args:
        broadband_method: 'weighted' (default), 'mean', or 'geomean'
    """
    
    if len(all_participant_data) == 0:
        print("No data to plot!")
        return
    
    print(f"\n{'='*60}")
    print(f"Creating BROADBAND ONLY plot for {len(all_participant_data)} participants...")
    print(f"Method: {broadband_method}")
    print(f"{'='*60}\n")
    
    # Calculate grid dimensions
    n_participants = len(all_participant_data)
    n_rows = int(np.ceil(n_participants / n_cols))
    
    # Create figure
    fig_width = n_cols * 6
    fig_height = n_rows * 4
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height))
    axes = np.atleast_2d(axes).flatten()
    
    # Plot each participant
    for plot_idx, participant_data in enumerate(all_participant_data):
        ax = axes[plot_idx]
        
        subject_id = participant_data['subject_id']
        session_id = participant_data['session_id']
        seizure_data = participant_data['seizure_data']
        n_valid_channels = participant_data['n_valid_channels']
        n_files_used = participant_data.get('n_files_used', '?')
        
        n_seizures = len(seizure_data)
        
        # Different colors for different seizures
        seizure_colors = plt.cm.Set1(np.linspace(0, 1, min(n_seizures, 9)))
        if n_seizures > 9:
            seizure_colors = plt.cm.tab20(np.linspace(0, 1, n_seizures))
        
        for seizure_idx_pos, (seizure_idx, band_dict) in enumerate(seizure_data.items()):
            # Compute broadband power
            time_array, broadband_power = compute_broadband_power(band_dict, method=broadband_method)
            
            if len(time_array) == 0:
                continue
            
            label = f'Seizure {seizure_idx_pos + 1}'
            
            ax.plot(time_array, broadband_power,
                   color=seizure_colors[seizure_idx_pos],
                   linestyle='-',
                   alpha=0.8,
                   linewidth=2.5,
                   label=label)
        
        # Mark seizure time
        ax.axvline(0, color='red', linestyle='--', linewidth=2, alpha=0.5, zorder=0)
        
        # Formatting
        ax.set_xlabel('Time relative to seizure (min)', fontsize=9)
        ylabel = f'Broadband Power (μV²/Hz) [{broadband_method}]'
        if log_scale:
            ylabel += ' [log scale]'
        ax.set_ylabel(ylabel, fontsize=9)
        
        title = f'Sub-{subject_id}, Ses-{session_id}\n{n_seizures} seizure(s), {n_valid_channels} channels, {n_files_used} files'
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        # Set scale
        if log_scale:
            ax.set_yscale('log')
        
        # Add legend if multiple seizures
        if n_seizures > 1:
            ax.legend(loc='upper right', fontsize=7, framealpha=0.9)
    
    # Remove empty subplots
    for ax in axes[len(all_participant_data):]:
        ax.remove()
    
    # Overall title
    scale_text = ' [Log Scale]' if log_scale else ''
    method_text = {
        'weighted': 'Bandwidth-Weighted',
        'mean': 'Arithmetic Mean',
        'geomean': 'Geometric Mean'
    }.get(broadband_method, broadband_method)
    
    fig.suptitle(f'Broadband Power Around Seizure Events ({method_text}){scale_text} (All Participants)\n'
                 f'Averaged across valid channels | ±{window_minutes} min windows',
                 fontsize=14, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    
    # Save plot
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    suffix = '_log' if log_scale else '_linear'
    output_file = Path(output_dir) / f'all_participants_seizure_epochs_broadband_{broadband_method}{suffix}.png'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"\nSaved: {output_file}")
    plt.show()
    plt.close()

In [10]:
# Load file metadata
file_metadata = load_file_metadata('/oak/stanford/groups/ckeller1/data/iEEG_EHR/iEEG_NWB/sherlock_file_registry.csv')

# Load all participant data (this takes time, but only do it once)
all_participant_data = load_all_participants_data(
    file_metadata=file_metadata,
    window_minutes=30
)

NameError: name 'load_file_metadata' is not defined

In [ ]:
# Create plot - can run multiple times with different parameters
# Option 3: Plot one band at a time
plot_all_participants_single_band(
    all_participant_data=all_participant_data,
    band_name='theta',
    output_dir='./seizure_plots'
)

In [ ]:
# Example 1: Plot broadband ONLY (recommended to start)
plot_broadband_only(
    all_participant_data,
    broadband_method='weighted',  # Options: 'weighted', 'mean', 'geomean'
    log_scale=True
)

In [11]:
#!/usr/bin/env python3
"""
analyze_seizure_epochs_broadband.py
Extract and visualize broadband power (averaged across all frequency bands and channels) 
around seizure events from iEEG data.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pynwb import NWBHDF5IO
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
BASE_PATH = Path('/oak/stanford/groups/ckeller1/data/iEEG_EHR/iEEG_NWB')
WINDOW_MINUTES = 7  # Minutes before and after seizure

# Frequency bands (indices in the band_power data)
BAND_NAMES = ['delta', 'theta', 'alpha', 'beta', 'gamma', 'high_gamma']

# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

def load_file_metadata(metadata_csv_path: str = None) -> pd.DataFrame:
    """
    Load the file metadata CSV.
    
    Args:
        metadata_csv_path: Path to the metadata CSV file. If None, looks for it in BASE_PATH.
    
    Returns:
        DataFrame with file metadata
    """
    if metadata_csv_path is None:
        # Look for CSV in base path
        metadata_csv_path = BASE_PATH / 'file_metadata.csv'
    
    metadata = pd.read_csv(metadata_csv_path)
    
    # Convert datetime columns
    metadata['start_datetime'] = pd.to_datetime(metadata['start_datetime'])
    metadata['end_datetime'] = pd.to_datetime(metadata['end_datetime'])
    
    # Only keep rows with preprocessed files
    metadata = metadata[metadata['has_preprocessed'] == True].copy()
    
    print(f"Loaded metadata for {len(metadata)} preprocessed files")
    
    return metadata


def get_all_subjects_with_seizures(file_metadata: pd.DataFrame) -> List[Tuple[str, str]]:
    """
    Find all subject-session pairs that have both preprocessed files and seizure data.
    
    Returns:
        List of (subject_id, session_id) tuples
    """
    subjects_sessions = []
    
    # Get unique subject-session combinations from metadata
    unique_sessions = file_metadata[['sub_id', 'ses_id']].drop_duplicates()
    
    for _, row in unique_sessions.iterrows():
        subject_id = row['sub_id'].replace('sub-', '')
        session_id = row['ses_id'].replace('ses-', '')
        
        # Check if seizure file exists
        ehr_dir = BASE_PATH / f'sub-{subject_id}' / f'ses-{session_id}' / 'ehr'
        seizure_file = ehr_dir / f'sub-{subject_id}_ses-{session_id}_seizures.csv'
        
        if seizure_file.exists():
            subjects_sessions.append((subject_id, session_id))
            
            # Count files for this session
            n_files = len(file_metadata[
                (file_metadata['sub_id'] == f'sub-{subject_id}') & 
                (file_metadata['ses_id'] == f'ses-{session_id}')
            ])
            print(f"Found: sub-{subject_id}, ses-{session_id} ({n_files} files)")
    
    return subjects_sessions


def load_seizure_times(subject_id: str, session_id: str) -> pd.DataFrame:
    """Load seizure times from EHR folder."""
    ehr_dir = BASE_PATH / f'sub-{subject_id}' / f'ses-{session_id}' / 'ehr'
    seizure_file = ehr_dir / f'sub-{subject_id}_ses-{session_id}_seizures.csv'
    
    if not seizure_file.exists():
        raise FileNotFoundError(f"Seizure file not found: {seizure_file}")
    
    # Load seizure times
    seizures = pd.read_csv(seizure_file)
    seizures['date'] = pd.to_datetime(seizures['date'])
    
    # Filter for this specific subject and session
    seizures = seizures[(seizures['sub_id'] == f'sub-{subject_id}') & 
                        (seizures['ses_id'] == f'ses-{session_id}')]
    
    return seizures


def get_relevant_files(file_metadata: pd.DataFrame,
                      subject_id: str,
                      session_id: str,
                      seizure_df: pd.DataFrame,
                      window_minutes: int = 7) -> List[Path]:
    """
    Get only the NWB files that overlap with seizure windows.
    
    Args:
        file_metadata: DataFrame with file metadata
        subject_id: Subject ID
        session_id: Session ID
        seizure_df: DataFrame with seizure times
        window_minutes: Window size around seizures
    
    Returns:
        List of Path objects for relevant NWB files
    """
    # Filter metadata for this session
    session_files = file_metadata[
        (file_metadata['sub_id'] == f'sub-{subject_id}') & 
        (file_metadata['ses_id'] == f'ses-{session_id}')
    ].copy()
    
    if len(session_files) == 0:
        return []
    
    relevant_files = []
    
    for seizure_idx, seizure_row in seizure_df.iterrows():
        seizure_time = seizure_row['date']
        
        # Calculate window
        window_start = seizure_time - pd.Timedelta(minutes=window_minutes)
        window_end = seizure_time + pd.Timedelta(minutes=window_minutes)
        
        # Find files that overlap with this window
        overlapping = session_files[
            (session_files['start_datetime'] <= window_end) &
            (session_files['end_datetime'] >= window_start)
        ]
        
        for _, file_row in overlapping.iterrows():
            file_path = Path(file_row['preprocessed_file_path'])
            if file_path not in relevant_files:
                relevant_files.append(file_path)
    
    return sorted(relevant_files)


def load_electrode_labels(nwb_file_path: Path) -> Dict[int, str]:
    """Load electrode labels from NWB file."""
    electrode_labels = {}
    try:
        with NWBHDF5IO(str(nwb_file_path), 'r') as io:
            nwb = io.read()
            electrodes_df = nwb.electrodes.to_dataframe()
            
            if 'Desikan_Killiany_cathode' in electrodes_df.columns:
                for idx, row in electrodes_df.iterrows():
                    label = row['Desikan_Killiany_cathode']
                    if pd.isna(label) or label == '' or label == 'nan':
                        electrode_labels[idx] = f'Ch{idx}'
                    else:
                        electrode_labels[idx] = str(label)
    except Exception as e:
        print(f"Warning: Could not load electrode labels: {e}")
    
    return electrode_labels


def filter_valid_channels(electrode_labels: Dict[int, str]) -> Dict[int, str]:
    """Filter out channels with 'unknown' or 'white-matter' in their labels."""
    valid_labels = {}
    
    for ch_idx, label in electrode_labels.items():
        label_lower = label.lower()
        if 'unknown' not in label_lower and 'white-matter' not in label_lower and 'white matter' not in label_lower:
            valid_labels[ch_idx] = label
    
    return valid_labels


def extract_broadband_seizure_epochs(nwb_file: Path, 
                                     seizure_df: pd.DataFrame,
                                     valid_channel_indices: set,
                                     window_minutes: int = 7) -> Dict:
    """
    Extract broadband power (averaged across all channels AND all bands).
    
    Returns:
        dict: {seizure_idx: {'times': [time_arrays], 'powers': [power_arrays]}}
    """
    seizure_data = {}
    
    try:
        with NWBHDF5IO(str(nwb_file), 'r') as io:
            nwb = io.read()
            
            # Get session start time
            session_start = nwb.session_start_time
            
            # Load band power data
            band_power_series = nwb.processing['ecephys']["band_power"]
            band_power_data = band_power_series.data[:]  # Shape: (timepoints, channels, bands)
            
            # Get timestamps
            if band_power_series.timestamps is not None:
                band_power_timestamps = band_power_series.timestamps[:]
            else:
                starting_time = band_power_series.starting_time
                rate = band_power_series.rate
                n_timepoints = band_power_data.shape[0]
                band_power_timestamps = starting_time + np.arange(n_timepoints) / rate
            
            # Convert timestamps to datetime
            band_power_datetimes = pd.to_datetime(session_start) + pd.to_timedelta(band_power_timestamps, unit='s')
            band_power_datetimes = band_power_datetimes.tz_localize(None)
            
            # Iterate through each seizure event
            for seizure_idx, row in seizure_df.iterrows():
                seizure_time = row['date']
                
                # Calculate window
                window_start = seizure_time - pd.Timedelta(minutes=window_minutes)
                window_end = seizure_time + pd.Timedelta(minutes=window_minutes)
                
                # Find indices within the window
                mask = (band_power_datetimes >= window_start) & (band_power_datetimes <= window_end)
                
                if not np.any(mask):
                    continue
                
                # Get data for this window
                window_data = band_power_data[mask, :, :]  # Shape: (time_in_window, channels, bands)
                window_times = band_power_datetimes[mask]
                
                # Filter to valid channels only
                valid_channel_list = sorted(list(valid_channel_indices))
                window_data_filtered = window_data[:, valid_channel_list, :]  # Shape: (time, valid_channels, bands)
                
                # Average across channels AND bands (broadband)
                avg_broadband = np.mean(window_data_filtered, axis=(1, 2))  # Shape: (time,)
                
                # Convert times to minutes relative to seizure
                time_relative = (window_times - seizure_time).total_seconds() / 60.0
                time_relative = time_relative.values
                
                # Store or accumulate data
                if seizure_idx not in seizure_data:
                    seizure_data[seizure_idx] = {'times': [], 'powers': []}
                
                seizure_data[seizure_idx]['times'].append(time_relative)
                seizure_data[seizure_idx]['powers'].append(avg_broadband)
            
    except Exception as e:
        print(f"  Error processing {nwb_file.name}: {e}")
        return {}
    
    return seizure_data


def load_all_participants_broadband_data(file_metadata: pd.DataFrame,
                                        window_minutes: int = 7) -> List[Dict]:
    """
    Load broadband power data for all participants.
    
    Returns:
        List of dicts with participant data
    """
    
    print(f"\n{'='*60}")
    print(f"Finding all subjects with seizure data...")
    print(f"{'='*60}\n")
    
    # Find all subjects
    subjects_sessions = get_all_subjects_with_seizures(file_metadata)
    
    if len(subjects_sessions) == 0:
        print("No subjects found with preprocessed data and seizure files!")
        return []
    
    print(f"\nFound {len(subjects_sessions)} subject-session pairs")
    print(f"{'='*60}\n")
    
    # Load data for all subjects
    all_participant_data = []
    
    for subject_id, session_id in subjects_sessions:
        print(f"Loading sub-{subject_id}, ses-{session_id}...")
        
        # Load seizure times
        seizure_df = load_seizure_times(subject_id, session_id)
        
        if len(seizure_df) == 0:
            print(f"  No seizure events found")
            continue
        
        # Get only relevant files that overlap with seizure windows
        relevant_files = get_relevant_files(
            file_metadata, subject_id, session_id, seizure_df, window_minutes
        )
        
        if len(relevant_files) == 0:
            print(f"  No relevant files found")
            continue
        
        print(f"  Found {len(relevant_files)} relevant files")
        
        # Load and filter electrode labels from first file
        electrode_labels = load_electrode_labels(relevant_files[0])
        valid_electrode_labels = filter_valid_channels(electrode_labels)
        valid_channel_indices = set(valid_electrode_labels.keys())
        
        n_valid_channels = len(valid_channel_indices)
        
        if n_valid_channels == 0:
            print(f"  No valid channels")
            continue
        
        # Collect broadband data across relevant files only
        all_seizure_data = {}
        
        for nwb_file in relevant_files:
            file_data = extract_broadband_seizure_epochs(
                nwb_file, seizure_df, valid_channel_indices, window_minutes
            )
            
            # Merge data
            for seizure_idx, data_dict in file_data.items():
                if seizure_idx not in all_seizure_data:
                    all_seizure_data[seizure_idx] = {'times': [], 'powers': []}
                
                all_seizure_data[seizure_idx]['times'].extend(data_dict['times'])
                all_seizure_data[seizure_idx]['powers'].extend(data_dict['powers'])
        
        # Concatenate data for each seizure
        for seizure_idx in all_seizure_data.keys():
            times_list = all_seizure_data[seizure_idx]['times']
            powers_list = all_seizure_data[seizure_idx]['powers']
            
            if len(times_list) > 0:
                all_times = np.concatenate(times_list)
                all_powers = np.concatenate(powers_list)
                
                # Sort by time
                sort_idx = np.argsort(all_times)
                all_seizure_data[seizure_idx] = (all_times[sort_idx], all_powers[sort_idx])
            else:
                all_seizure_data[seizure_idx] = (np.array([]), np.array([]))
        
        if len(all_seizure_data) > 0:
            all_participant_data.append({
                'subject_id': subject_id,
                'session_id': session_id,
                'seizure_data': all_seizure_data,
                'seizure_df': seizure_df,
                'n_valid_channels': n_valid_channels,
                'n_files_used': len(relevant_files)
            })
            print(f"  Loaded {len(all_seizure_data)} seizure(s), {n_valid_channels} valid channels")
        else:
            print(f"  No valid seizure data")
    
    print(f"\n{'='*60}")
    print(f"Successfully loaded data for {len(all_participant_data)} participants")
    print(f"{'='*60}\n")
    
    return all_participant_data


# ============================================================================
# PLOTTING FUNCTION
# ============================================================================

def plot_all_participants_broadband(all_participant_data: List[Dict],
                                   output_dir: str = './plots',
                                   window_minutes: int = 7,
                                   n_cols: int = 3,
                                   log_scale: bool = True):
    """
    Plot all participants' broadband power (averaged across all channels and bands) around seizures.
    
    Args:
        all_participant_data: List of dicts with participant data
        output_dir: Directory to save plots
        window_minutes: Window size (for title)
        n_cols: Number of columns in subplot grid
        log_scale: If True, use log scale for y-axis (default: True)
    """
    
    if len(all_participant_data) == 0:
        print("No data to plot!")
        return
    
    print(f"\n{'='*60}")
    print(f"Creating broadband plot for {len(all_participant_data)} participants...")
    print(f"{'='*60}\n")
    
    # Calculate grid dimensions
    n_participants = len(all_participant_data)
    n_rows = int(np.ceil(n_participants / n_cols))
    
    # Create figure
    fig_width = n_cols * 6
    fig_height = n_rows * 4
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height))
    axes = np.atleast_2d(axes).flatten()
    
    # Plot each participant
    for plot_idx, participant_data in enumerate(all_participant_data):
        ax = axes[plot_idx]
        
        subject_id = participant_data['subject_id']
        session_id = participant_data['session_id']
        seizure_data = participant_data['seizure_data']
        n_valid_channels = participant_data['n_valid_channels']
        n_files_used = participant_data.get('n_files_used', '?')
        
        n_seizures = len(seizure_data)
        
        # Color map for seizures
        colors = plt.cm.Set1(np.linspace(0, 1, min(n_seizures, 9)))
        if n_seizures > 9:
            colors = plt.cm.tab20(np.linspace(0, 1, n_seizures))
        
        # Plot each seizure
        for color_idx, (seizure_idx, (time_array, power_array)) in enumerate(seizure_data.items()):
            if len(time_array) > 0:
                ax.plot(time_array, power_array,
                       color=colors[color_idx],
                       alpha=0.7,
                       linewidth=2,
                       label=f'Seizure {color_idx + 1}')
        
        # Mark seizure time
        ax.axvline(0, color='red', linestyle='--', linewidth=2, alpha=0.5)
        
        # Formatting
        ax.set_xlabel('Time relative to seizure (min)', fontsize=9)
        ylabel = 'Broadband Power (μV²/Hz)'
        if log_scale:
            ylabel += ' [log scale]'
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(f'Sub-{subject_id}, Ses-{session_id}\n{n_seizures} seizure(s), {n_valid_channels} channels, {n_files_used} files',
                    fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        if log_scale:
            ax.set_yscale('log')
        
        # Add legend if multiple seizures
        if n_seizures > 1:
            ax.legend(loc='upper right', fontsize=7, framealpha=0.9)
    
    # Remove empty subplots
    for ax in axes[len(all_participant_data):]:
        ax.remove()
    
    # Overall title
    scale_text = ' [Log Scale]' if log_scale else ''
    fig.suptitle(f'Broadband Power Around Seizure Events{scale_text} (All Participants)\n'
                 f'Averaged across all valid channels and frequency bands | ±{window_minutes} min windows',
                 fontsize=14, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    
    # Save plot
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    suffix = '_log' if log_scale else '_linear'
    output_file = Path(output_dir) / f'all_participants_seizure_broadband{suffix}.png'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"\nSaved: {output_file}")
    plt.show()
    plt.close()